<a href="https://colab.research.google.com/github/Ivory-Uche/Portfolio-Pilot/blob/main/Portfolio_pilot_M3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PortfolioPilot - Milestone 3: cleaning, EDA, dimension reduction, feature set



**Run the cells in order, top to bottom.** Each cell uses variables made by the cells before it.
After a kernel restart, start again from cell 1.

Rule for M3: the test sets are created in Step 3 and then never explored. Every statistic, cap,
table and figure uses **training data only**.

In [3]:
%pip install numpy pandas scipy scikit-learn matplotlib yfinance

## Step 1 - Setting up

In [10]:
from google.colab import drive
drive.mount("/content/drive")          # a pop-up asks you to allow access: click Allow

import os
os.chdir("/content/drive/MyDrive/Portfolio_pilot/data/raw")
print("Working in:", os.getcwd())
print("Files here:", os.listdir())

from pathlib import Path
SCF_FILE = Path("SCFP2025.csv")
OUT_DIR  = Path("outputs")


SEED = 7275

# SCF column names in the summary extract (check against the SCF codebook)
COLS = {
    "household_id": "YY1", "record_id": "Y1", "weight": "WGT", "age": "AGE",
    "income": "INCOME", "networth": "NETWORTH", "liq": "LIQ", "debt": "DEBT",
    "equity": "EQUITY", "fin": "FIN", "yes_risk": "YESFINRISK", "no_risk": "NOFINRISK",
}
IMPLICATE = 1
SPLIT = {"train": 0.70, "val": 0.15, "test": 0.15}
CAP_QUANTILE = 0.99            # caps for ratio features, from training data

K_RANGE = [2, 3, 4, 5, 6]
CLUSTER_METHOD = "kmeans"      # "kmeans" or "gmm"
CHOSEN_K = None                # None = automatic (best silhouette); or set a number after Step 9
MIN_CLUSTER_SHARE = 0.05
N_BOOTSTRAP = 20               # use 3 for a quick test, 20 for final results
PCA_THRESHOLD = 0.80

TICKERS = ["SPY", "VEA", "VWO", "AGG", "TLT", "TIP", "VNQ", "GLD", "DBC"]
ETF_START, ETF_END = "2016-01-01", "2026-01-01"   # end is exclusive: data through 2025-12-31
ETF_TRAIN_END, ETF_VAL_END = "2021-12-31", "2023-12-31"
ETF_FILE = OUT_DIR / "raw" / "etf_prices_raw.csv"

FIG_DIR, TAB_DIR, DATA_DIR = OUT_DIR / "figures", OUT_DIR / "tables", OUT_DIR / "processed"
for d in [FIG_DIR, TAB_DIR, DATA_DIR, ETF_FILE.parent]:
    d.mkdir(parents=True, exist_ok=True)

print("SCF file found:", SCF_FILE.exists())
print("Outputs go to:", OUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in: /content/drive/MyDrive/Portfolio_pilot/data/raw
Files here: ['SCFP2025.csv', 'PortfolioPilot_ETF_Prices.csv', 'outputs']
SCF file found: True
Outputs go to: outputs


## Step 2 - Import and helpers

In [13]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})
np.random.seed(SEED)

NUMBERS = {}   # every count quoted in the report is stored here and saved in the last step
def note(section, key, value):
    NUMBERS.setdefault(section, {})[key] = value

def to_plain(v):
    """Make numpy values JSON-friendly."""
    if isinstance(v, dict):
        return {str(k): to_plain(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [to_plain(x) for x in v]
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, (np.floating, float)):
        return round(float(v), 6)
    return v

def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

def weighted_resample(df, seed):
    """Draw households with probability proportional to the survey weight."""
    rng = np.random.default_rng(seed)
    p = df["weight"].to_numpy(float)
    return df.iloc[rng.choice(len(df), size=len(df), replace=True, p=p / p.sum())]

def weighted_median(x, w):
    o = np.argsort(np.asarray(x))
    x, w = np.asarray(x, float)[o], np.asarray(w, float)[o]
    c = np.cumsum(w)
    return float(x[np.searchsorted(c, 0.5 * c[-1])])

print("Ready")

Ready


## Step 3 - Load the SCF and check its structure

In [14]:
raw = pd.read_csv(SCF_FILE, low_memory=False)
raw.columns = [c.upper() for c in raw.columns]
note("raw", "rows", len(raw))
note("raw", "columns", raw.shape[1])

missing_cols = [c for c in COLS.values() if c not in raw.columns]
assert not missing_cols, f"Columns not in the file: {missing_cols}. Fix COLS in Step 1."

full = raw[list(COLS.values())].copy()
full.columns = list(COLS.keys())
full["implicate"] = (full["record_id"] % 10).astype(int)    # Y1 = YY1*10 + implicate
note("raw", "households", full["household_id"].nunique())
note("raw", "implicates", sorted(full["implicate"].unique().tolist()))
del raw

print(NUMBERS["raw"])
full.head()

{'rows': 21800, 'columns': 362, 'households': 4360, 'implicates': [1, 2, 3, 4, 5]}


,household_id,record_id,weight,age,income,networth,liq,debt,equity,fin,yes_risk,no_risk,implicate
0,1,11,1241.701712,39,317351.35135,2434270.0,80300,250000.0,120150.0,215300,0,0,1
1,1,12,1247.038677,39,317351.35135,2208400.0,80300,250000.0,113800.0,229300,0,0,2
2,1,13,1234.428835,39,317351.35135,2128300.0,80300,250000.0,75300.0,209300,0,0,3
3,1,14,1267.530314,39,317351.35135,1742300.0,80300,250000.0,72800.0,212300,0,0,4
4,1,15,911.408367,39,317351.35135,1517300.0,80300,250000.0,73600.0,228300,0,0,5


## Step 4 - Cleaning: duplicates, missing values, impossible values

In [15]:
note("duplicates", "exact_duplicate_rows", int(full.duplicated().sum()))
note("duplicates", "duplicate_record_ids", int(full["record_id"].duplicated().sum()))
full = full.drop_duplicates(subset="record_id")

hh = full[full["implicate"] == IMPLICATE].copy()           # one row per household
note("implicate_1", "rows", len(hh))

na = hh.isna().sum()
note("missing", "by_column", na[na > 0].to_dict())
note("missing", "total", int(na.sum()))

checks = {
    "age_under_18_or_over_100": (hh["age"] < 18) | (hh["age"] > 100),
    "weight_not_positive": hh["weight"] <= 0,
    "liq_negative": hh["liq"] < 0,
    "debt_negative": hh["debt"] < 0,
    "equity_negative": hh["equity"] < 0,
    "fin_negative": hh["fin"] < 0,
    "equity_more_than_fin": hh["equity"] > hh["fin"] + 1,
    "liq_more_than_fin": hh["liq"] > hh["fin"] + 1,
    "both_risk_flags_1": (hh["yes_risk"] == 1) & (hh["no_risk"] == 1),
}
for name, mask in checks.items():
    note("errors", name, int(mask.sum()))
bad = checks["age_under_18_or_over_100"] | checks["weight_not_positive"] | checks["both_risk_flags_1"]
note("errors", "rows_dropped", int(bad.sum()))
hh = hh[~bad].reset_index(drop=True)

# Real values that need special handling later (not errors)
for name, mask in {"income_zero": hh["income"] == 0, "income_negative": hh["income"] < 0,
                   "networth_negative": hh["networth"] < 0, "fin_zero": hh["fin"] == 0,
                   "debt_zero": hh["debt"] == 0}.items():
    note("genuine", name, int(mask.sum()))

# Risk willingness: 0 = no risk, 1 = average, 2 = above-average / substantial
hh["risk_willingness"] = np.select([hh["no_risk"] == 1, hh["yes_risk"] == 1], [0, 2], default=1)
hh["age_band"] = pd.cut(hh["age"], [0, 34, 44, 54, 64, 74, 200],
                        labels=["<35", "35-44", "45-54", "55-64", "65-74", "75+"]).astype(str)
note("implicate_1", "rows_after_cleaning", len(hh))

for s in ["duplicates", "implicate_1", "missing", "errors", "genuine"]:
    print(s, NUMBERS[s])

duplicates {'exact_duplicate_rows': 0, 'duplicate_record_ids': 0}
implicate_1 {'rows': 4360, 'rows_after_cleaning': 4360}
missing {'by_column': {}, 'total': 0}
errors {'age_under_18_or_over_100': 0, 'weight_not_positive': 0, 'liq_negative': 0, 'debt_negative': 0, 'equity_negative': 0, 'fin_negative': 0, 'equity_more_than_fin': 0, 'liq_more_than_fin': 0, 'both_risk_flags_1': 0, 'rows_dropped': 0}
genuine {'income_zero': 23, 'income_negative': 0, 'networth_negative': 330, 'fin_zero': 53, 'debt_zero': 1115}


## Step 5 - Split train / validation / test (before any exploration)
Stratified on age band x risk willingness. Split by household, so implicates 2-5 can follow the same IDs later.


In [16]:
strata = hh["age_band"] + "|" + hh["risk_willingness"].astype(str)
counts = strata.value_counts()
strata = strata.where(strata.map(counts) >= 10, "other")      # merge tiny strata
note("split", "n_strata", strata.nunique())

train_val, test = train_test_split(hh, test_size=SPLIT["test"], stratify=strata, random_state=SEED)
train, val = train_test_split(train_val, test_size=SPLIT["val"] / (SPLIT["train"] + SPLIT["val"]),
                              stratify=strata.loc[train_val.index], random_state=SEED)
parts = {"train": train.reset_index(drop=True), "val": val.reset_index(drop=True),
         "test": test.reset_index(drop=True)}

for name, d in parts.items():
    note("split", f"{name}_households", len(d))
    note("split", f"{name}_weight_share", d["weight"].sum() / hh["weight"].sum())
pd.concat([pd.DataFrame({"household_id": d["household_id"], "partition": n}) for n, d in parts.items()]) \
  .to_csv(DATA_DIR / "scf_split_ids.csv", index=False)
NUMBERS["split"]

{'n_strata': 18,
 'train_households': 3052,
 'train_weight_share': np.float64(0.7033403995202065),
 'val_households': 654,
 'val_weight_share': np.float64(0.1441687614754587),
 'test_households': 654,
 'test_weight_share': np.float64(0.15249083900433474)}

## Step 6 - Implicate structure check

In [19]:
tr_full = full[full["household_id"].isin(parts["train"]["household_id"])]
g = tr_full.groupby("household_id")
note("implicates", "train_households_without_5_rows", int((g.size() != 5).sum()))
for c in ["income", "networth", "liq", "debt"]:
    note("implicates", f"share_where_{c}_varies", float((g[c].nunique() > 1).mean()))
NUMBERS["implicates"]

{'train_households_without_5_rows': 0,
 'share_where_income_varies': 0.6055045871559633,
 'share_where_networth_varies': 0.9121887287024901,
 'share_where_liq_varies': 0.4384010484927916,
 'share_where_debt_varies': 0.42660550458715596}